### Using Claude

In [1]:
from dotenv import load_dotenv
import os
import warnings

from langchain_anthropic import ChatAnthropic

warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv("/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI/.env")

prompt = "In what year was the first Star Wars movie released?"


def create_claude_llm(model_name="claude-sonnet-4-6", **kwargs):

    api_key = os.getenv("ANTHROPIC_API_KEY")

    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY not found")

    llm = ChatAnthropic(
        model=model_name,
        anthropic_api_key=api_key,
        **kwargs
    )

    return llm


# Test it
llm_claude = create_claude_llm()

response = llm_claude.invoke(prompt)

print(response.content)

The first Star Wars movie, *Star Wars: Episode IV – A New Hope*, was released in **1977**.


### LLM Chains

**Chain vs No Chain**

Chains refer to sequences of calls — whether to an LLM, a tool, or a data preprocessing step.

- **Syntax**: Chain uses `prompt → llm → parser` piping; manual uses separate `.invoke()` calls passed step by step.
- **Readability**: Chain is compact and idiomatic; manual is verbose but shows each step explicitly.
- **Debugging**: Chain hides intermediate values; manual lets you print/inspect `formatted_prompt` and `llm_response` at each stage.
- **Use case**: Chain is for production code and real chains; manual is for teaching/learning what's happening under the hood.
- **Extensibility**: Chain makes it easy to add more steps; manual requires more boilerplate per added step.
- **LangChain idiom**: Chain is the standard students will see in docs; manual is rarely written this way in practice.

In [ ]:
# Done - With Chains
from langchain.prompts import PromptTemplate

# Template stays exactly the same - prompt construction is provider-agnostic
superbowl_template = "who won super bowl in year {year}?"

prompt_template = PromptTemplate(
    template=superbowl_template,
    input_variables=["year"]
)

from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser

# Claude's chat model wrapper - needs ANTHROPIC_API_KEY set in your env
# (e.g. via os.environ["ANTHROPIC_API_KEY"] or a .env file loaded with python-dotenv)
llm_1 = ChatAnthropic(
    model="claude-sonnet-4-6",   # swap in claude-opus-4-7 or claude-haiku-4-5-20251001 if needed
    temperature=0
)

# prompt_template -> formats your input dict/value into the prompt string
# llm_1            -> sends that prompt to Claude, gets back a ChatMessage
# StrOutputParser  -> extracts just the plain text string from the response
simple_chain = prompt_template | llm_1 | StrOutputParser()

# Both invocation styles work identically to before
response = simple_chain.invoke(1977)

# response = simple_chain.invoke({"year": 1977})

response

"There was **no Super Bowl played in 1977**. However:\n\n- **Super Bowl XI** was played on **January 9, 1977**, and the **Oakland Raiders** defeated the Minnesota Vikings **32–14**.\n- **Super Bowl XII** was played on **January 15, 1978**, and the **Dallas Cowboys** defeated the Denver Broncos **27–10**.\n\nSo if you're referring to the Super Bowl played **in January 1977**, the **Oakland Raiders** won!"

In [ ]:
# Done - Without Chains
from langchain.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser

# Template stays exactly the same - prompt construction is provider-agnostic
superbowl_template = "who won super bowl in year {year}?"

prompt_template = PromptTemplate(
    template=superbowl_template,
    input_variables=["year"]
)

# Claude's chat model wrapper - needs ANTHROPIC_API_KEY set in your env
llm_1 = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0
)

output_parser = StrOutputParser()

# Step 1: format the prompt template into an actual prompt string/object
formatted_prompt = prompt_template.invoke({"year": 1977})
# formatted_prompt is a PromptValue - print(formatted_prompt.to_string())
# -> "who won super bowl in year 1977?"

# Step 2: pass the formatted prompt to Claude manually
llm_response = llm_1.invoke(formatted_prompt)
# llm_response is an AIMessage object
# llm_response.content -> "The Oakland Raiders won Super Bowl XI..."

# Step 3: parse the AIMessage down to a plain string
response = output_parser.invoke(llm_response)

response

## 3. Simple Sequential Chain

Use **Chain-1** to answer a question. Then send the question/answer pair to **Chain-2** to verify if the answer is correct or not.

https://api.python.langchain.com/en/stable/chains/langchain.chains.sequential.SimpleSequentialChain.html#langchain.chains.sequential.SimpleSequentialChain

### Create a LLMChain - 1

In [3]:
# Template for chain 1 simply say define a term

llm_claude = create_claude_llm()

prompt_template_1 = PromptTemplate.from_template(
    template="define {term}", 
    inputs=['term']
)

# Create the LLM chain object - This method of chain creation is now deprecated
# llm_chain_1 = LLMChain(
#     llm = llm_1,
#     prompt = prompt_template_1,
#     output_key = "definition"
# )

llm_chain_1 = prompt_template_1 | llm_claude 

## example
llm_chain_1.invoke("momentum")

AIMessage(content="# Momentum\n\n## Definition\n\n**Momentum** is the quantity of motion an object possesses, determined by its **mass** and **velocity**.\n\n## Formula\n\n$$\\vec{p} = m\\vec{v}$$\n\n| Symbol | Meaning | Unit |\n|--------|---------|------|\n| **p** | Momentum | kg·m/s |\n| **m** | Mass | kilograms (kg) |\n| **v** | Velocity | m/s |\n\n## Key Properties\n\n- It is a **vector quantity** (has both magnitude and direction)\n- Direction of momentum = direction of velocity\n- A **heavier** or **faster** object has greater momentum\n\n## Types\n\n- **Linear Momentum** – motion in a straight line\n- **Angular Momentum** – momentum of rotating objects\n\n## Law of Conservation of Momentum\n\n> *In a closed system with no external forces, total momentum remains **constant***.\n\n$$p_{before} = p_{after}$$\n\n## Simple Example\n\nA 2 kg ball moving at 3 m/s has momentum:\n$$p = 2 \\times 3 = 6 \\text{ kg·m/s}$$\n\n## Connection to Force\n\n$$\\vec{F} = \\frac{\\Delta \\vec{p}}{\\

### Create the LLMChain-2

In [4]:
# Template for OpenAI
prompt_template_2 = PromptTemplate.from_template(
    template="""rate the accuracy of the definition on a scale of (0 to 5).
                Definition of :  {definition}""", 
    inputs=['definition']
)

## The following template if used with SimpleSequentialChain will FAIL
## Added for demonstration purposes only - to try uncomment the code below
# prompt_template_openai =PromptTemplate.from_template(template="""
# rate the accuracy of the definition on a scale of (0 to 5).
# Definition of {term}:  {definition}""", inputs=['definition','term'])

llm_claude = create_claude_llm()

# This method of creating chains is now deprecated
# llm_chain_2 = LLMChain(
#     llm = llm_2,
#     prompt = prompt_template_2,
#     output_key = "score"
# )

llm_chain_2 = prompt_template_2 | llm_claude

llm_chain_2

PromptTemplate(input_variables=['definition'], input_types={}, partial_variables={}, template='rate the accuracy of the definition on a scale of (0 to 5).\n                Definition of :  {definition}')
| ChatAnthropic(model='claude-sonnet-4-6', anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

In [ ]:
# https://api.python.langchain.com/en/latest/chains/langchain.chains.sequential.SequentialChain.html
from langchain.chains import SimpleSequentialChain
from operator import itemgetter

# This method of creating a sequential chain is now deprecated
# simple_sequential_chain = SimpleSequentialChain(
#     chains=[llm_chain_1, llm_chain_2],
#     verbose = True,
# )

simple_sequential_chain = ({
        "definition" : llm_chain_1
    } | llm_chain_2
)

In [ ]:
# Input = String value if there is ONLY one input variable
# Input = Dictionary if there are multiple input variables

response = simple_sequential_chain.invoke("moment of interia")

print(response)